In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.gridspec as gridspec
from sklearn.metrics import matthews_corrcoef, confusion_matrix
from scipy.stats import fisher_exact
import polars as pl
from pathlib import Path

In [ ]:
folders = ["../Results/HepG2_individual_results", "../Results/K562_individual_results"]

combined = pl.concat([
    pl.read_csv(csv)
    for folder in folders
    for csv in Path(folder).glob("*.csv")
], how="vertical_relaxed")

combined = combined.sort(["Feature", "index"])

combined.write_csv("../Results/CRISPR_final_results.csv")
print(combined.shape)

In [ ]:
path = ('../Results/CRISPR_final_results.csv')

In [ ]:
table = pd.read_csv(path)

## Generate scatterplot for one feature with JUST CTRL SHAP

In [ ]:
features = ['PRPF8_5_binding', 'EIF4E_2_binding']

In [ ]:
def one_feature_scatter(data, features):
    
    for feature in features:
    
        plt.style.use(
            "../../../paper.mplstyle"
        )
    
        # ------------------------------------------------------------------
        # 1. Subset & significance filter
        # ------------------------------------------------------------------
        cell_line = 'K562'
        
        feature_subset = data[
            (data['cell_line'] == cell_line) &
            (data['Feature'] == feature)
        ].copy()
    
        sig_mask = (
            (feature_subset['dPSI'].abs() > 0) &
            (feature_subset['FDR'] <= 0.1)
        )
    
        # ------------------------------------------------------------------
        # 2. Plot
        # ------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(8, 5))
    
        ax.scatter(
            feature_subset.loc[~sig_mask, 'CTRL_SHAP'],
            feature_subset.loc[~sig_mask, 'dPSI'],
            color='gainsboro',
            alpha=0.5,
            s=30,
            label='Nonsignificant', 
            rasterized=True
        )
    
        ax.scatter(
            feature_subset.loc[sig_mask, 'CTRL_SHAP'],
            feature_subset.loc[sig_mask, 'dPSI'],
            color='steelblue',
            alpha=0.5,
            s=30,
            label='Significant', 
            rasterized=True
        )
    
        # ------------------------------------------------------------------
        # 3. Axes formatting
        # ------------------------------------------------------------------
        ax.axhline(y=0, color='black', linestyle='-', linewidth=1.5)
        ax.axvline(x=0, color='black', linestyle='-', linewidth=1.5)
        ax.grid(True, alpha=0.3)
        ax.set_xlabel(r"Local SHAP", fontsize=18, labelpad=4, fontweight='bold')
        ax.set_ylabel(r"$\Delta\psi$", fontsize=18, labelpad=0, fontweight='bold')
        ax.tick_params(axis='both', labelsize=12)
    
        parts = feature.split('_')
        rbp_name = parts[0]
        position = parts[1] if len(parts) > 1 else ''
        ax.set_title(f"{rbp_name} Position {position} — {cell_line}", fontsize=22, pad=10)
    
        ax.legend(
            loc='upper left',
            bbox_to_anchor=(1, 1),
            frameon=False,
            fontsize=18,
            markerscale=1.5
        )
    
        plt.tight_layout()
        plt.savefig(f"{feature}_scatterplot.pdf", bbox_inches='tight', dpi=200)
        plt.show()

In [ ]:
one_feature_scatter(table, features)